### Este notebook é executado via Lakeflow job em resources/pipeline.job.yml

Realiza o download dos arquivos zip da Receita Federal, da base pública dos dados abertos do Governo Federal.
- Fonte: https://dados.gov.br/dados/conjuntos-dados/cadastro-nacional-da-pessoa-juridica---cnpj

In [ ]:
import os
import time
import urllib.parse
import xml.etree.ElementTree as ET
import zipfile
import requests
from urllib3.exceptions import IncompleteRead, ProtocolError

# Configurações do link compartilhado
BASE_URL = dbutils.widgets.get("BASE_URL")
SHARE_TOKEN = dbutils.widgets.get("SHARE_TOKEN")
TARGET_DIR = dbutils.widgets.get("TARGET_DIR")
OUTPUT_FOLDER = dbutils.widgets.get("OUTPUT_FOLDER")

# Endpoint WebDAV público do Nextcloud
WEBDAV_URL = f"{BASE_URL}/public.php/webdav{TARGET_DIR}"

try:
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    # 1. Lista os arquivos da pasta via requisição WebDAV (PROPFIND)
    response = requests.request(
        method="PROPFIND",
        url=WEBDAV_URL,
        auth=(SHARE_TOKEN, ""),
        headers={"Depth": "1"}
    )
    response.raise_for_status()

    # 2. Faz o parse do XML retornado para filtrar apenas os arquivos .zip
    root = ET.fromstring(response.content)
    namespace = {"d": "DAV:"}
    zip_files = []

    for elem in root.findall("d:response", namespace):
        href = elem.find("d:href", namespace).text
        decoded_href = urllib.parse.unquote(href)
        if decoded_href.lower().endswith(".zip"):
            filename = os.path.basename(decoded_href)
            zip_files.append((filename, f"{BASE_URL}{href}"))

    print(f"Encontrados {len(zip_files)} arquivos .zip.\n")

    MAX_RETRIES = 5

    for filename, file_url in zip_files:
        filepath = os.path.join(OUTPUT_FOLDER, filename)
        temp_filepath = f"{filepath}.tmp"
        csv_final_path = os.path.join(OUTPUT_FOLDER, f"{os.path.splitext(filename)[0]}.csv")

        # Pula apenas se o arquivo final (.zip ou .csv) já existir completo
        if os.path.exists(filepath) or os.path.exists(csv_final_path):
            print(f"[PULANDO] {filename} já existe.")
            continue

        sucesso_download = False

        for tentativa in range(1, MAX_RETRIES + 1):
            try:
                print(f"[BAIXANDO] {filename} (Tentativa {tentativa}/{MAX_RETRIES})...")
                
                if os.path.exists(temp_filepath):
                    os.remove(temp_filepath)

                with requests.get(file_url, auth=(SHARE_TOKEN, ""), stream=True, timeout=(30, 120)) as r:
                    r.raise_for_status()
                    with open(temp_filepath, "wb") as f:
                        for chunk in r.iter_content(chunk_size=1024 * 1024):  # Chunks de 1 MB
                            if chunk:
                                f.write(chunk)

                # Só renomeia para .zip definitivo após concluir 100% do download
                os.rename(temp_filepath, filepath)
                print(f"[CONCLUÍDO] {filename}")
                sucesso_download = True
                break

            except (requests.exceptions.RequestException, IncompleteRead, ProtocolError, OSError) as e:
                print(f"[AVISO] Queda de conexão ao baixar {filename}: {e}")
                if os.path.exists(temp_filepath):
                    os.remove(temp_filepath)
                
                if tentativa < MAX_RETRIES:
                    espera = tentativa * 10
                    print(f"Aguardando {espera}s antes de tentar novamente...")
                    time.sleep(espera)
                else:
                    print(f"[ERRO] Falha definitiva ao baixar {filename} após {MAX_RETRIES} tentativas.")

        if not sucesso_download:
            continue

        # 4. Descompacta o arquivo .zip
        try:
            print(f"[DESCOMPACTANDO] {filename}...")
            with zipfile.ZipFile(filepath, 'r') as zip_ref:
                zip_ref.extractall(OUTPUT_FOLDER)
                print(f"[DESCOMPACTADO] {filename} para {OUTPUT_FOLDER}")

                extracted_files = zip_ref.namelist()
                for extracted_file in extracted_files:
                    extracted_file_path = os.path.join(OUTPUT_FOLDER, extracted_file)
                    os.rename(extracted_file_path, csv_final_path)
                    print(f"[RENOMEADO] {extracted_file} para {csv_final_path}\n")

        except zipfile.BadZipFile as e:
            print(f"[ERRO] {filename} está corrompido e será removido: {e}")
            if os.path.exists(filepath):
                os.remove(filepath)
        except OSError as e:
            print(f"[ERRO] Falha ao descompactar/renomear {filename}: {e}")

except requests.exceptions.RequestException as e:
    print(f"Erro na requisição inicial (PROPFIND): {e}")
except OSError as e:
    print(f"Erro ao criar diretório de destino: {e}")